In [1]:
import mcelect
import awkward as ak
import numpy as np
import hvplot.pandas  # noqa
import pandas as pd
import panel as pn
import config

In [12]:
mw_config = {
    "voters": 600,
    "candidates": 12,
    "mode": "multi_winner",
    "committee_size": 5,
    "considerations": [{"electorate": {
        "dimensions": 2,
        "distance_function": { "q_gaussian_2": 1.0},
        "factions": [
            { "popularity" : 1.0, "voter_center": [-2.0, 1.0], "voter_spread": 1.0, "candidate_spread": 0.5 },
            { "popularity" : 0.95, "voter_center": [2.0, 1.0], "voter_spread": 1.0, "candidate_spread": 0.5 },
            { "popularity" : 0.5, "voter_center": [-2.0, -1.0], "voter_spread": 1.0, "candidate_spread": 0.5 },
            # { "popularity" : 1.0, "voter_center": [2.0, -1.0], "voter_spread": 1.0, "candidate_spread": 0.5 },
        ]
    }}],
    "committee_methods": [
        {"PluralityTopN": {}},
        { "RRV": { "strat": "Honest", "ranks": 10, "k": 0.5} },
    ]
}

In [ ]:
table = mcelect.simulate(mw_config, 10000)
results = ak.from_arrow(table)

In [14]:
results.methods.rrv_10_h.winners

<Array [[0, 10, 2, 5, 11], ..., [0, 4, 3, 7, 1]] type='10000 * 5 * uint32'>

In [15]:
rrv_winners = ak.from_regular(results.methods.rrv_10_h.winners, axis=1)
pltn_winners = ak.from_regular(results.methods.pltn.winners, axis=1)

In [16]:
rrv_factions = ak.flatten(results.electorate.faction[rrv_winners])
pltn_factions = ak.flatten(results.electorate.faction[pltn_winners])
rrv_positions = ak.flatten(results.electorate.positions[rrv_winners])
pltn_positions = ak.flatten(results.electorate.positions[pltn_winners])

In [17]:
factions_df = pd.DataFrame(dict(
    rrv_factions=rrv_factions,
    pltn_factions=pltn_factions,
    rrv_pos_x=rrv_positions[:,0],
    rrv_pos_y=rrv_positions[:,1],
    pltn_pos_x=pltn_positions[:,0],
    pltn_pos_y=pltn_positions[:,1],
))

In [18]:
pn.Column(
    factions_df.hvplot.hist(y="rrv_factions"),
    factions_df.hvplot.hist(y="pltn_factions"),
)

BokehModel(combine_events=True, render_bundle={'docs_json': {'4e1588e9-ff5f-47af-8031-d6989fae781c': {'version…

In [19]:
factions_df.hvplot.scatter(x="rrv_pos_x", y="rrv_pos_y", color="rrv_factions")

:Scatter   [rrv_pos_x]   (rrv_pos_y,rrv_factions)

In [20]:
factions_df.hvplot.scatter(x="pltn_pos_x", y="pltn_pos_y", color="pltn_factions")

:Scatter   [pltn_pos_x]   (pltn_pos_y,pltn_factions)

In [2]:
cfg = config.Config(voters=600, candidates=12, considerations=[config.Consideration(likability=config.Likability(mean=1.0))])

In [6]:
print(cfg.model_dump_json(exclude_none=True))

{"voters":600,"candidates":12,"considerations":[{"likability":{"mean":1.0}}]}
